In [ ]:
import wfdb
import matplotlib.pyplot as plt

record = wfdb.rdrecord('../data/raw/mit-bih-arrhythmia-database-1.0.0/202')       # read record 100 (file 100.dat + 100.hea)
annotation = wfdb.rdann('../data/raw/mit-bih-arrhythmia-database-1.0.0/202', 'atr')

# Plot ECG signal (first 10 seconds)
plt.plot(record.p_signal[:3600,0])  # Channel 0, first 10 seconds (360 Hz * 10 sec)
plt.title('ECG signal - Record 3500')
plt.xlabel('Samples')
plt.ylabel('Amplitude')
plt.show()


In [ ]:
import wfdb
import numpy as np
import pandas as pd

# Load record and annotations
record = wfdb.rdrecord('../data/raw/mit-bih-arrhythmia-database-1.0.0/202')       # read record 100 (file 100.dat + 100.hea)
annotation = wfdb.rdann('../data/raw/mit-bih-arrhythmia-database-1.0.0/202', 'atr')

fs = record.fs
win_pre = int(0.2 * fs)  # 200ms before beat
win_post = int(0.4 * fs)  # 400ms after beat

beats = []
labels = []

for i, sample in enumerate(annotation.sample):
    start = sample - win_pre
    end = sample + win_post
    if start >= 0 and end <= len(record.p_signal):
        beat = record.p_signal[start:end, 0]  # first channel
        beats.append(beat)
        labels.append(annotation.symbol[i])

beats = np.array(beats)
labels = np.array(labels)

# Create DataFrame: each row = one beat, columns = amplitude points
df = pd.DataFrame(beats)
df['label'] = labels

# Save to CSV
df.to_csv('../data/processed/05/ecg_beats_record_100.csv', index=False)

print("Saved ECG beats and labels to ecg_beats_record_100.csv")


In [ ]:
import pandas as pd

df = pd.read_csv('../data/processed/05/ecg_beats_record_100.csv')

print(df.head())
print(df['label'].value_counts())  # Check how many beats per class

print(df.shape)  # Rows = beats, Columns = 217 (216 samples + 1 label)


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

print(dict(zip(le.classes_, le.transform(le.classes_))))
# Example output: {'N': 2, 'V': 3, 'A': 0, 'L': 1} (depends on your data)


In [ ]:
X = df.drop(columns=['label', 'label_enc'])
y = df['label_enc']


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_, labels=np.unique(y_test)))


the precision is low because the data is imbalanced .. the normal N dominates. this hurts the rare classes

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

clf = RandomForestClassifier(class_weight='balanced', random_state=42)
clf = GradientBoostingClassifier()  # another good option
clf = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')  # if using XGBoost

clf.fit(X_train, y_train)


In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

# Filter out classes with fewer than 2 samples
class_counts = y.value_counts()
valid_classes = class_counts[class_counts >= 2].index
X_filtered = X[y.isin(valid_classes)]
y_filtered = y[y.isin(valid_classes)]

# First split your data into train and test
X_train, X_test, y_train, y_test = train_test_split(X_filtered, y_filtered, test_size=0.2, random_state=42, stratify=y_filtered)

# Apply RandomUnderSampler on the training data only
rus = RandomUnderSampler(random_state=42)
X_train_res, y_train_res = rus.fit_resample(X_train, y_train)

print("Original training shape:", X_train.shape)
print("Resampled training shape:", X_train_res.shape)


In [ ]:
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=le.classes_, labels=np.unique(y_test)))


we will be using Hyperparameter tuning

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Bidirectional, Dense, Dropout, Flatten, BatchNormalization

model = Sequential([
    Conv1D(64, kernel_size=5, activation='relu', input_shape=(input_length, 1)),
    MaxPooling1D(pool_size=2),
    BatchNormalization(),

    Conv1D(128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    BatchNormalization(),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),

    Bidirectional(LSTM(64)),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')  # or sigmoid for binary
])
